In [1]:
print(123)

123


In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")


In [3]:
df_ground_truth.head()

,question,document
0,Can I enroll in this course if I'm just findin...,74eb249bbf
1,What's the deadline for submitting my project ...,74eb249bbf
2,Do I need to start the course at a specific ti...,74eb249bbf
3,"If I join late, am I still able to complete ev...",74eb249bbf
4,Is there a cutoff date for when I have to turn...,74eb249bbf


In [4]:
ground_truth = df_ground_truth.to_dict(orient="records")

In [5]:
ground_truth[10]

{'question': 'Where can I find the link to watch the office hours live?',
 'document': '489dd1c9d9'}

In [6]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [7]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [8]:
text_search("What is the difference between LLM and traditional machine learning?")

[{'id': '57c8d3a1e4',
  'course': 'llm-zoomcamp',
  'section': 'Module 4: Evaluation',
  'question': 'Why is there a gap between lessons 6 and 11 in Module 4?',
  'answer': 'No lessons are missing. Module 4 is split into two subsections, and the filenames reflect that structure:\n\n- `01`–`06`: RAG evaluation\n- `11`–`15`: agent evaluation\n\nYou can read the numbering as lessons 0.1–0.6 for the first subsection and 1.1–1.5 for the second. That is why the filenames jump from `06-search-tuning.md` to `11-evaluation-intro.md`.'},
 {'id': 'dbf5369006',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Can I use Bluesky for learning in public credits?',
  'answer': 'Yes. Bluesky posts can be used for learning in public credits.'},
 {'id': 'db78580409',
  'course': 'llm-zoomcamp',
  'section': 'Module 2: Vector Search',
  'question': 'What is the cosine similarity?',
  'answer': 'Cosine similarity is a measure used to calculate the similarity betwee

In [9]:
q = ground_truth[10]
q

{'question': 'Where can I find the link to watch the office hours live?',
 'document': '489dd1c9d9'}

In [10]:
doc_id = q['document']
doc_id

'489dd1c9d9'

In [11]:
results = text_search(q['question'])
results

[{'id': '489dd1c9d9',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'What is the video/zoom link to the stream for the “Office Hours” or live/workshop sessions?',
  'answer': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be missed if the room is very active.'},
 {'id': 'd65e05bd7a',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Are there live sessions or office hours for each module?',
  'answer': 'There are no separate live sessions for every module by default. Module materials are pre-recorded and

In [12]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

489dd1c9d9 == 489dd1c9d9: True
d65e05bd7a == 489dd1c9d9: False
53f15299b6 == 489dd1c9d9: False
930286278d == 489dd1c9d9: False
5cc511f85b == 489dd1c9d9: False


In [13]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[1, 0, 0, 0, 0]

In [14]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [15]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [16]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [17]:
relevance_total_text[:15]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [18]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance 

In [19]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [20]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [21]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/765 [00:00<?, ?it/s]

In [22]:
relevance_total[:15]

[[1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [23]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [24]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

In [25]:
cnt / len(example)
# 0.933

0.9333333333333333

In [26]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [27]:
hit_rate(relevance_total)

0.7895424836601307

In [28]:
example[1]
# [0, 1, 0, 0, 0]

[0, 1, 0, 0, 0]

In [29]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.333333333333332

In [30]:
total_score / len(example)
# 0.822

0.8222222222222222

In [31]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [32]:
mrr(relevance_total)
# 0.822

0.6545533769063173

In [33]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [34]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/765 [00:00<?, ?it/s]

{'hit_rate': 0.7895424836601307, 'mrr': 0.6545533769063173}

In [35]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    ) 

In [36]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/765 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8366013071895425, 'mrr': 0.7133551198257077}


  0%|          | 0/765 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8261437908496732, 'mrr': 0.7105228758169928}


  0%|          | 0/765 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.7895424836601307, 'mrr': 0.6545533769063173}


  0%|          | 0/765 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.7699346405228759, 'mrr': 0.6243355119825704}


  0%|          | 0/765 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.742483660130719, 'mrr': 0.5979738562091497}


In [37]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [38]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/765 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/765 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/765 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/765 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/765 [00:00<?, ?it/s]

In [39]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.905882,0.783987
19,2.0,4.0,0.2,0.905882,0.783987
35,5.0,10.0,0.5,0.905882,0.783987
18,2.0,4.0,0.1,0.905882,0.782876
34,5.0,10.0,0.2,0.904575,0.782505
33,5.0,10.0,0.1,0.901961,0.781983
4,1.0,2.0,0.2,0.907190,0.779325
7,1.0,4.0,0.2,0.916340,0.778475
20,2.0,4.0,0.5,0.905882,0.776449
6,1.0,4.0,0.1,0.913725,0.775425
